# Attention 精简版:naive → compact 速查

> 本 notebook 是第 4 章的速查卡。完整推导见 [`ch04.ipynb`](./ch04.ipynb)。

## 完整的 shape 流程图

minimind 注意力层的张量形状变化全流程(`b=2, T=5, d=768, n_q=8, n_kv=4, d_h=96`):

```
输入 x:        (b, T, d)           = (2, 5, 768)
  │ Wq/Wk/Wv
  ▼ view
Q:             (b, T, n_q, d_h)    = (2, 5, 8, 96)
K,V:           (b, T, n_kv, d_h)   = (2, 5, 4, 96)
  │ QK-Norm (on d_h)
  │ RoPE (on d_h)
  │ repeat_kv (K,V: n_kv→n_kv*n_rep=n_q)
  │ transpose(1,2)
  ▼
Q,K,V:         (b, n_h, T, d_h)    = (2, 8, 5, 96)
  │ Q @ K^T / √d_h
  ▼
scores:        (b, n_h, T, T)      = (2, 8, 5, 5)
  │ + causal mask
  │ softmax
  ▼
attn weights:  (b, n_h, T, T)      = (2, 8, 5, 5)
  │ @ V
  ▼
out:           (b, n_h, T, d_h)    = (2, 8, 5, 96)
  │ transpose(1,2) + reshape
  ▼
out:           (b, T, n_h*d_h)     = (2, 5, 768)
  │ Wo
  ▼
输出:          (b, T, d)           = (2, 5, 768)  ← 和输入一样!
```

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# === naive self-attention(3 行核心） ===
def attention_naive(Q, K, V):
    # Q,K,V: (T, d)
    scores = Q @ K.T / (Q.shape[-1] ** 0.5)  # (T,d)@(d,T)->(T,T)
    return F.softmax(scores, dim=-1) @ V      # (T,T)@(T,d)->(T,d)

x = torch.randn(3, 4)
print("naive:", attention_naive(x, x, x).shape)  # (3, 4)

In [ ]:
# === compact: minimind 风格的完整单头注意力 ===
def attention_compact(x, Wq, Wk, Wv, Wo):
    # x: (b, T, d)
    Q, K, V = Wq(x), Wk(x), Wv(x)              # (b,T,d) -> (b,T,d)
    d_k = Q.shape[-1]
    scores = Q @ K.transpose(-2, -1) / d_k**0.5 # (b,T,d)@(b,d,T)->(b,T,T)
    # causal mask
    T = x.shape[1]
    mask = torch.full((T, T), float('-inf')).triu(1)  # (T,T)
    attn = F.softmax(scores + mask, dim=-1)     # (b,T,T)
    return Wo(attn @ V)                          # (b,T,T)@(b,T,d)->(b,T,d)->(b,T,d)

b, T, d = 2, 5, 8
x = torch.randn(b, T, d)
Wq = nn.Linear(d, d, bias=False)
Wk = nn.Linear(d, d, bias=False)
Wv = nn.Linear(d, d, bias=False)
Wo = nn.Linear(d, d, bias=False)
print("compact:", attention_compact(x, Wq, Wk, Wv, Wo).shape)  # (2, 5, 8)

## 关键公式速记

$$\text{Attention}(Q,K,V) = \text{softmax}\!\left(\frac{QK^T}{\sqrt{d_k}} + M_{\text{causal}}\right)V$$

$$\text{RoPE}(x, m) = x \cdot \cos(m\theta) + \text{rotate\_half}(x) \cdot \sin(m\theta)$$

$$\text{GQA: } K_{\text{full}} = \text{repeat\_kv}(K_{n_{kv}},\; n_{rep}) \quad \text{where } n_{rep} = n_q / n_{kv}$$